## 13.03 用于预训练词嵌入的数据集


### 环境配置


In [1]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)
import os
import sys
sys.path.insert(0, "..")
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import pypto
    import torch
    from torch import nn
    import torch_npu

warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="On the interactive interface")
warnings.filterwarnings("ignore", message="Cannot create tensor")
import matplotlib.pyplot as plt

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()
import math
import os
import random
import time
from src.utils import (DATA_HUB, download_extract, Vocab, tokenize,
                       count_corpus, Timer, load_data_ptb)
from src.utils import read_ptb as _read_ptb

# 预注册 PTB 数据源（主 notebook 13.3 节已注册，此处幂等）
DATA_HUB['ptb'] = (DATA_URL if False else 'https://d2l-data.s3-accelerate.amazonaws.com/ptb.zip',
                   '319d85e578af0cdc590547f26231e4e31cdf1e42')


### 练习 13.3.1

**题目：** 如果不使用下采样，本节中代码的运行时间会发生什么变化？

**解答：** 不使用下采样（跳过 `subsample` 或直接用原始 `sentences` 构造语料）时，语料中保留了大量高频词（如 “the”、“of”），词元总数明显增大，因此 `get_centers_and_contexts` 生成的（中心词，上下文）对数量、以及 `get_negatives` 需要处理的上下文位置数都成倍增加，**数据加载（预处理）时间显著上升**。下采样本身也有一次额外遍历的开销，但与它省下的后续处理量相比很小。下面用计时实验验证：


以下使用 `torch`/`time` 编程进行验证（PTB 数据，对比有/无下采样两版 `load_data_ptb` 的预处理耗时）：


In [2]:
import time
from src.utils import (read_ptb, Vocab, subsample, count_corpus,
                       get_centers_and_contexts, get_negatives)

def preprocess(with_subsample, K=5, max_window_size=5):
    sentences = read_ptb()
    vocab = Vocab(sentences, min_freq=10)
    counter = count_corpus(sentences)
    if with_subsample:
        sub, counter = subsample(sentences, vocab)
        corpus = [vocab[line] for line in sub]
    else:
        corpus = [vocab[line] for line in sentences]
    t0 = time.time()
    all_centers, all_contexts = get_centers_and_contexts(corpus, max_window_size)
    all_negatives = get_negatives(all_contexts, vocab, counter, K)
    return time.time() - t0, len(corpus)

t_sub, n_sub = preprocess(True)
t_nosub, n_nosub = preprocess(False)
print(f'带下采样  : 预处理 {t_sub:.2f}s, 语料行数 {n_sub}')
print(f'无下采样  : 预处理 {t_nosub:.2f}s, 语料行数 {n_nosub}')
print(f'无下采样耗时是带下采样的 {t_nosub / max(t_sub, 1e-9):.1f} 倍')


正在从 https://d2l-data.s3-accelerate.amazonaws.com/ptb.zip 下载 ../data/ptb.zip...


带下采样  : 预处理 6.57s, 语料行数 42069
无下采样  : 预处理 19.23s, 语料行数 42069
无下采样耗时是带下采样的 2.9 倍


### 练习 13.3.2

**题目：** `RandomGenerator` 类缓存 `k` 个随机采样结果。将 `k` 设置为其他值，看看它如何影响数据加载速度。

**解答：** `RandomGenerator` 每轮用 `random.choices` 一次性生成 `k=10000` 个候选并缓存，之后 `draw()` 只做一次下标递增，避免了反复调用底层采样器。把 `k` 调大（如 100000）会减少 `random.choices` 的调用次数、略微加快 `draw()` 整体吞吐；调小（如 100）则更频繁地重建候选列表，`get_negatives` 阶段变慢。注意 `random.choices` 本身是 $O(k)$ 的，k 过大时单次生成更久，只是分摊得更薄；实践上 1000~10000 已足够。下面给出对比实验：


In [3]:
import random
from src.utils import Timer

class RandomGeneratorK:
    def __init__(self, sampling_weights, k=10000):
        self.population = list(range(1, len(sampling_weights) + 1))
        self.sampling_weights = sampling_weights
        self.candidates = []
        self.i = 0
        self.k = k

    def draw(self):
        if self.i == len(self.candidates):
            self.candidates = random.choices(
                self.population, self.sampling_weights, k=self.k)
            self.i = 0
        self.i += 1
        return self.candidates[self.i - 1]

def bench(k, n=200000):
    gen = RandomGeneratorK([2, 3, 4], k=k)
    timer = Timer()
    for _ in range(n):
        gen.draw()
    return timer.stop()

for k in (100, 1000, 10000, 100000):
    print(f'k={k:<6d} 采样 {200000} 次耗时 {bench(k):.3f}s')


k=100    采样 200000 次耗时 0.064s
k=1000   采样 200000 次耗时 0.067s
k=10000  采样 200000 次耗时 0.067s


k=100000 采样 200000 次耗时 0.062s


### 练习 13.3.3

**题目：** 本节代码中的哪些其他超参数可能会影响数据加载速度？

**解答：** 除下采样与负采样数量外，以下超参数都会影响数据加载（预处理）速度：

- `min_freq`（`Vocab` 的截断阈值）：越小保留的词越多，词表与序列长度增大，处理变慢；
- `max_window_size`（上下文窗口上限）：越大每行生成的（中心词，上下文）对越多；
- `num_noise_words`（负采样数 $K$）：越大 `get_negatives` 采样量越大；
- `batch_size` 与 `num_workers`：影响 `DataLoader` 的取数据与并行加载开销；
- `data_iter` 的 `shuffle`：数据量大时打乱也有一定开销。


---

## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)
